In [ ]:
from pathlib import Path
import re, unicodedata
import pandas as pd
from ruamel.yaml import YAML
import nbformat as nbf  # chỉ dùng khi OUTPUT_FORMAT="ipynb"
import os
from pathlib import Path

def remove_content_after_parts(file_path):
    try:
        # Đọc toàn bộ nội dung của file
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()

        # Tìm vị trí dòng chứa "parts:"
        parts_index = None
        for i, line in enumerate(lines):
            if line.strip() == "parts:":
                parts_index = i
                break

        # Nếu không tìm thấy "parts:", báo lỗi và kết thúc
        if parts_index is None:
            print("Không tìm thấy dòng chứa 'parts:' trong file.")
            return

        # Giữ lại các dòng trước "parts:" và xóa phần còn lại
        new_content = lines[:(parts_index+1)]

        # Ghi nội dung mới vào file
        with open(file_path, 'w', encoding='utf-8') as file:
            file.writelines(new_content)

    except Exception as e:
        print(f"Lỗi khi xử lý file: {e}")

def delete_md_files(folder_path):
    # Kiểm tra xem folder_path có tồn tại không
    if not os.path.exists(folder_path):
        print(f"Thư mục '{folder_path}' không tồn tại.")
        return

    # Lặp qua tất cả các file trong thư mục
    for filename in os.listdir(folder_path):
        # Kiểm tra nếu file có đuôi .md
        if filename.endswith(".md"):
            file_path = os.path.join(folder_path, filename)
            try:
                # Xóa file
                os.remove(file_path)
                print(f"Đã xóa: {file_path}")
            except Exception as e:
                print(f"Lỗi khi xóa file {file_path}: {e}")

# ========== CẤU HÌNH ==========
BOOK_ROOT = Path.cwd()

BOOK_ROOT      = Path.cwd() # Path(r"D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong")  # thư mục chứa _config.yml, _toc.yml
EXCEL_PATH     = Path.cwd()/"Ke_hoach_cong_viec.xlsx"
folder_path    = Path.cwd()/"chapters"
SHEET_NAME     = 0                    # hoặc "Sheet1"
OUTPUT_FORMAT  = "md"                 # "md" hoặc "ipynb"
DEFAULT_GROUP  = "Khác"               # caption mặc định nếu ô nhóm trống
# =================================

# Tên cột (chấp nhận nhiều biến thể; script sẽ tìm cột đầu tiên khớp)
COLS = {
    "group":      ["Nhóm công việc", "Nhóm", "Group"],
    "title":      ["Tên công việc"],
    "status":     ["Tiến độ hoàn thành", "Trạng thái", "Status"],
    "desc":       ["Mô tả công việc", "Mô tả", "Công việc chi tiết"],
    "done":       ["Các công việc đã làm", "Đã làm", "Kết quả đã đạt", "Công việc đúng tiến độ", "Các công việc đã làm"],
    "todo":       ["Những công việc cần hoàn thiện", "Cần hoàn thiện", "Việc cần làm thêm", "Công việc chậm tiến độ", "Những công việc cần hoàn thiện"],
    "kk_chuquan": ["Khó khăn chủ quan", "Khó khăn (chủ quan)", "Khó khăn vướng mắc chủ quan"],
    "kk_khachquan":["Khó khăn khách quan", "Khó khăn (khách quan)", "Khó khăn vướng mắc khách quan"],
}

CHAPTERS_DIR   = BOOK_ROOT / "chapters"
TOC_PATH       = BOOK_ROOT / "_toc.yml"
CHAPTERS_DIR.mkdir(parents=True, exist_ok=True)

def slugify(text: str) -> str:
    txt = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")
    txt = re.sub(r"[^A-Za-z0-9]+", "-", txt.strip()).strip("-").lower()
    return txt or "item"

def pick_col(cols_available, candidates):
    for c in candidates:
        if c in cols_available:
            return c
    return None

def get_val(row, candidates, default=""):
    for c in candidates:
        if c in row and pd.notna(row[c]):
            v = str(row[c]).strip()
            if v != "":
                return v
    return default

def split_items(s: str):
    if not s:
        return []
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    parts = []
    for seg in re.split(r"\n|;", s):
        seg = seg.strip(" \t-•*")
        if seg:
            parts.append(seg)
    return parts

def md_bullets(items):
    return "\n".join(f"- {it}" for it in items) if items else "Không có"

# Icon dùng emoji (hiển thị ổn trên Jupyter Book)
ICONS = {
    "status": "📊",
    "desc": "📝",
    "done": "✅",
    "todo": "🧰",
    "issues": "⚠️",
    "subjective": "🧠",
    "objective": "🌐",
}

def status_badge(status: str | None) -> str:
    if not status or not str(status).strip():
        return "⚪ Chưa cập nhật"
    s = str(status).strip().lower()
    if any(k in s for k in ["đúng tiến độ", "on track", "đạt", "hoàn thành"]):
        return "🟢 Đúng tiến độ"
    if any(k in s for k in ["chậm", "trễ", "delay", "behind"]):
        return "🔴 Chậm tiến độ"
    if any(k in s for k in ["cần lưu ý", "rủi ro", "risk", "cảnh báo"]):
        return "🟡 Cần lưu ý"
    return f"🔘 {status.strip()}"

def render_markdown(title, status, desc, done_items, todo_items, kk_cq, kk_kq):
    return (
f"### {title}\n"
f"<hr>\n\n"
f"**Tình trạng:** {status_badge(status)}\n\n"
f"**Mô tả công việc:** {(desc or '—').strip()}\n\n"
f"{ICONS['done']} **Các công việc đã làm:**\n\n"
f"{md_bullets(done_items)}\n\n"
f"{ICONS['todo']} **Những công việc cần hoàn thiện:**\n\n"
f"{md_bullets(todo_items)}\n\n"
f"{ICONS['issues']} **Khó khăn vướng mắc:**\n\n"
f"{ICONS['subjective']} *Khó khăn chủ quan:*\n\n"
f"{(kk_cq or 'Không có').strip()}\n\n"
f"{ICONS['objective']} *Khó khăn khách quan:*\n\n"
f"{(kk_kq or 'Không có').strip()}\n"
    )

def write_md(path: Path, content: str):
    path.write_text(content, encoding="utf-8")

def write_ipynb(path: Path, content: str):
    nb = nbf.v4.new_notebook()
    nb.cells = [nbf.v4.new_markdown_cell(content)]
    with path.open("w", encoding="utf-8") as f:
        nbf.write(nb, f)

# 0) Xóa file .md hiện có
# Thay đổi đường dẫn đến thư mục của bạn

delete_md_files(folder_path)

# 1) Đọc Excel
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, dtype=str).fillna("")
if df.empty:
    raise SystemExit("Excel rỗng.")

# 2) Xác định cột thực tế theo từ điển COLS
cols_map = {}
for k, cand in COLS.items():
    found = pick_col(df.columns, cand)
    cols_map[k] = found

# 3) Load TOC
# Thay đổi đường dẫn đến file của bạn
file_path = TOC_PATH
remove_content_after_parts(file_path)

yaml = YAML()
yaml.preserve_quotes = True
if not TOC_PATH.exists():
    raise SystemExit(f"Không thấy {TOC_PATH}. Hãy kiểm tra BOOK_ROOT.")
with TOC_PATH.open("r", encoding="utf-8") as f:
    toc = yaml.load(f)

# Bảo đảm có parts
if "parts" not in toc or toc["parts"] is None:
    toc["parts"] = []

# Thu thập toàn bộ doc đã có để tránh trùng lặp (TOC lỗi nếu 1 doc xuất hiện nhiều nơi)
def collect_existing(obj) -> set[str]:
    s = set()
    if isinstance(obj, dict):
        if "file" in obj and isinstance(obj["file"], str):
            s.add(obj["file"])
        for v in obj.values():
            s |= collect_existing(v)
    elif isinstance(obj, list):
        for it in obj:
            s |= collect_existing(it)
    return s

existing_docs = collect_existing(toc)

# Lập chỉ mục các part theo caption hiện có
def ensure_part(caption: str):
    # Tìm part theo caption, nếu chưa có thì tạo
    for p in toc["parts"]:
        if p.get("caption") == caption:
            if "chapters" not in p or p["chapters"] is None:
                p["chapters"] = []
            return p
    new_p = {"caption": caption, "chapters": []}
    toc["parts"].append(new_p)
    return new_p

made = []

# 4) Tạo file cho từng dòng + thêm vào caption theo "Nhóm công việc"
for i, row in df.iterrows():
    group_name = get_val(row, COLS["group"], default=DEFAULT_GROUP)
    title      = get_val(row, COLS["title"], default=f"Mục {i+1}")
    status     = get_val(row, COLS["status"], default="")
    desc       = get_val(row, COLS["desc"],   default="")
    done_items = split_items(get_val(row, COLS["done"], default=""))
    todo_items = split_items(get_val(row, COLS["todo"], default=""))
    kk_cq      = get_val(row, COLS["kk_chuquan"],   default="")
    kk_kq      = get_val(row, COLS["kk_khachquan"], default="")

    # Tạo slug file
    slug_base = slugify(title)
    slug = slug_base
    k = 1
    while (CHAPTERS_DIR / f"{slug}.md").exists() or (CHAPTERS_DIR / f"{slug}.ipynb").exists():
        k += 1
        slug = f"{slug_base}-{k}"

    # Ghi nội dung
    content = render_markdown(title, status, desc, done_items, todo_items, kk_cq, kk_kq)
    out_path = CHAPTERS_DIR / (f"{slug}.md" if OUTPUT_FORMAT == "md" else f"{slug}.ipynb")
    if OUTPUT_FORMAT == "md":
        write_md(out_path, content)
    elif OUTPUT_FORMAT == "ipynb":
        write_ipynb(out_path, content)
    else:
        raise ValueError("OUTPUT_FORMAT phải là 'md' hoặc 'ipynb'.")

    # Thêm vào TOC dưới caption = group_name
    part = ensure_part(group_name if group_name.strip() else DEFAULT_GROUP)
    doc_entry = f"chapters/{slug}"  # không kèm đuôi
    if doc_entry not in existing_docs:
        part["chapters"].append({"file": doc_entry})
        existing_docs.add(doc_entry)
        made.append((group_name, doc_entry))

# 5) Lưu TOC
with TOC_PATH.open("w", encoding="utf-8") as f:
    yaml.dump(toc, f)

# 6) Thông báo
print(f"Tạo {len(made)} chapter mới, theo nhóm (caption) trong _toc.yml:")
for g, e in made[:10]:
    print(f"  [{g}] - {e}")
if len(made) > 10:
    print("  ...")
print("\nGiờ build sách:\n  jupyter-book build", BOOK_ROOT)

In [6]:
!jupyter-book build .

Running Jupyter-Book v1.0.4.post1
Source Folder: D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong
Config Path: D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\_config.yml
Output Path: D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\_build\html


[Warning] The `html.google_analytics_id` configuration value has moved to `html.analytics.google_analytics_id`


Running Sphinx v7.4.7
loading translations [en]... done
[etoc] Changing master_doc to 'README'
myst v3.0.1: MdParserConfig(commonmark_only=False, gfm_only=False, enable_extensions={'dollarmath', 'tasklist', 'colon_fence', 'substitution', 'linkify'}, disable_syntax=[], all_links_external=False, links_external_new_tab=False, url_schemes=('mailto', 'http', 'https'), ref_domains=None, fence_as_directive=set(), number_code_blocks=[], title_to_header=False, heading_anchors=0, heading_slug_func=None, html_meta={}, footnote_transition=True, words_per_minute=200, substitutions={}, lin

C:\Users\AD\AppData\Local\Programs\Python\Python313\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader family of methods required for zmq. Registering an additional selector thread for add_reader support via tornado. Use `asyncio.set_event_loop_policy(WindowsSelectorEventLoopPolicy())` to avoid this warning.
  self._get_loop()
D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\build book.ipynb: WARNING: Executing notebook failed: CellTimeoutError [mystnb.exec]
D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\build book.ipynb: WARNING: Notebook exception traceback saved in: D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\_build\html\reports\build book.err.log [mystnb.exec]
D:\Sach Khoa hoc du lieu trong Kinh te va Kinh doanh\report tu dong\chapters/ban-lien-lac-cuu-sinh-vien.md:1: WARNING: Document headings start at H3, not H1 [myst.header]
D:\Sach Khoa hoc du lieu trong Kinh te 